# 04 — Gerar diagnósticos e pares DPO

Objetivo: transformar tentativas validadas em entradas, diagnóstico estruturado e pares `chosen/rejected`. Entrada: `data/interim/erros_validados.jsonl`. Saída: `data/interim/diagnosticos_dpo.jsonl`. O diagnóstico é bastidor pedagógico; `resposta_esperada` é informação interna para o agente revisor. A chamada a LLM não é necessária nesta fase: regras transparentes geram registros reproduzíveis.

In [6]:
# Opcional: %pip install pydantic>=2.7
import json
from pathlib import Path
from typing import Any, Literal

from pydantic import BaseModel, ConfigDict

def localizar_projeto() -> Path:
    '''Encontra a pasta de dados sem estado compartilhado entre notebooks.'''
    for base in (Path.cwd(), *Path.cwd().parents):
        pasta = base / 'notebooks' / 'eja_diagnosticador_dataset'
        if (pasta / 'data' / 'config.json').exists(): return pasta
        if base.name == 'eja_diagnosticador_dataset' and (base / 'data' / 'config.json').exists(): return base
    raise FileNotFoundError('Configuração compartilhada não encontrada')

ROOT = localizar_projeto()
CONFIG = json.loads((ROOT / 'data' / 'config.json').read_text(encoding='utf-8'))
ENTRADA = ROOT / 'data' / 'interim' / 'erros_validados.jsonl'
SAIDA = ROOT / 'data' / 'interim' / 'diagnosticos_dpo.jsonl'
Acao = Literal['PERGUNTAR', 'DAR_DICA', 'VALIDAR_PARCIALMENTE', 'PROPOR_EXERCICIO']
TipoErro = Literal['ERRO_OPERACAO', 'ERRO_CALCULO', 'PASSO_OMITIDO', 'ERRO_LEITURA', 'SEM_ERRO']

class Modelo(BaseModel):
    '''Modelo de saída estrito; campos extras são rejeitados.'''
    model_config = ConfigDict(extra='forbid')

class Entrada(Modelo):
    duvida: str
    contexto_cotidiano: str
    historico_resumido: str

class Saida(Modelo):
    dificuldade: str
    verificacao_raciocinio: str
    acao: Acao
    resposta_esperada: str
    justificativa_acao: str

class Rubrica(Modelo):
    usou_contexto_cotidiano: bool
    evitou_infantilizar: bool
    linguagem_simples: bool
    nao_vazou_resposta: bool

class DPO(Modelo):
    chosen: str
    rejected: str

class Registro(Modelo):
    id: str
    base_id: str
    erro_tipo: TipoErro
    entrada: Entrada
    saida: Saida
    rubrica: Rubrica
    dpo: DPO
    modo_geracao: str

def serializar_jsonl(registro: dict[str, Any]) -> str:
    '''Escapa separadores Unicode para manter um registro por linha física.'''
    texto = json.dumps(registro, ensure_ascii=False)
    for caractere, escape in ((chr(0x85), r'\u0085'), (chr(0x2028), r'\u2028'), (chr(0x2029), r'\u2029')):
        texto = texto.replace(caractere, escape)
    return texto

def escolher_acao(tipo: str, tem_tentativa: bool = True) -> tuple[Acao, str]:
    '''Aplica critério explícito: sem tentativa pergunta; parcial valida; correto pratica; erro recebe dica.'''
    if not tem_tentativa: return 'PERGUNTAR', 'Ainda não há tentativa para analisar.'
    if tipo == 'PASSO_OMITIDO': return 'VALIDAR_PARCIALMENTE', 'Há avanço identificável, mas falta explicitar uma etapa.'
    if tipo == 'SEM_ERRO': return 'PROPOR_EXERCICIO', 'A solução está correta; um novo problema favorece prática autônoma.'
    return 'DAR_DICA', 'A tentativa contém uma operação, cálculo ou dado incorreto; a dica focaliza a revisão.'

def resumir_contexto(texto: str) -> str:
    '''Mantém uma referência curta ao cenário cotidiano sem copiar a descrição inteira.'''
    palavras = texto.split()[:6]
    conectores_finais = {'a', 'ao', 'aos', 'as', 'com', 'da', 'das', 'de', 'do', 'dos', 'e', 'em', 'entre', 'na', 'nas', 'no', 'nos', 'o', 'os', 'para', 'por', 'sobre', 'um', 'uma', 'uns', 'umas'}
    while palavras and palavras[-1].strip('.,;:').casefold() in conectores_finais:
        palavras.pop()
    resumo = ' '.join(palavras).strip(' .,;:')
    if resumo.split() and resumo.split()[0].casefold() in {'um', 'uma', 'o', 'a', 'os', 'as'}:
        resumo = resumo[0].lower() + resumo[1:]
    return resumo or 'situação do problema'

def gerar_registro(exemplo: dict[str, Any]) -> dict[str, Any]:
    '''Cria e valida um exemplo de diagnóstico com linguagem de apoio ao tutor.'''
    tipo = exemplo['erro_tipo']
    tentativa = exemplo['tentativa']
    acao, justificativa = escolher_acao(tipo)
    contexto = exemplo['contexto_cotidiano']
    contexto_referencia = resumir_contexto(contexto)
    resposta = exemplo['resposta_final']
    final = exemplo['resposta_numerica']
    verificacao = {
        'ERRO_OPERACAO': 'A operação escrita não representa a relação matemática indicada no enunciado.',
        'ERRO_CALCULO': 'A operação escolhida é adequada, mas o resultado informado não confere com a calculadora.',
        'ERRO_LEITURA': 'A tentativa usou um valor diferente do dado registrado no enunciado.',
        'PASSO_OMITIDO': 'Há uma parte do raciocínio, mas a etapa seguinte não foi apresentada.',
        'SEM_ERRO': 'As contas anotadas foram conferidas com a calculadora segura.'
    }[tipo]
    dificuldade = {
        'ERRO_OPERACAO': 'Escolha da operação matemática.',
        'ERRO_CALCULO': 'Precisão do cálculo aritmético.',
        'ERRO_LEITURA': 'Identificação dos dados relevantes do enunciado.',
        'PASSO_OMITIDO': 'Encadeamento incompleto do raciocínio.',
        'SEM_ERRO': 'Sem dificuldade observada nas contas verificadas.'
    }[tipo]
    if tipo == 'PASSO_OMITIDO':
        tentativa_texto = 'Já identifiquei alguns dados, mas não sei qual é o próximo passo.'
    else:
        tentativa_texto = f"Minha tentativa foi {tentativa['expressao_tentada']} = {tentativa['resultado_tentado']}. Estou no caminho certo?"
    pergunta = f"Problema: {exemplo['problema_localizado']}\n{tentativa_texto}"
    hist = 'Exemplo sintético; não corresponde ao histórico de uma pessoa real.'
    resposta_esperada = f'Resposta de referência para uso interno do revisor: {resposta}. Não mostrar ao estudante.'
    if acao == 'VALIDAR_PARCIALMENTE':
        chosen = f'Na situação "{contexto_referencia}", você já organizou parte do raciocínio. Qual etapa viria em seguida?'
    elif acao == 'PROPOR_EXERCICIO':
        chosen = f'Na situação "{contexto_referencia}", sua estratégia está consistente. Que outro exemplo parecido você gostaria de tentar?'
    elif acao == 'PERGUNTAR':
        chosen = f'Na situação "{contexto_referencia}", o que você já percebeu e qual seria o primeiro passo?'
    elif tipo == 'ERRO_OPERACAO':
        chosen = f'Na situação "{contexto_referencia}", identifique a relação entre as quantidades. Qual operação representa essa relação?'
    elif tipo == 'ERRO_CALCULO':
        chosen = f'Na situação "{contexto_referencia}", a operação faz sentido. Como você pode conferir o resultado por outro caminho?'
    else:
        chosen = f'Na situação "{contexto_referencia}", confira os valores do enunciado. Qual quantidade deve entrar neste passo?'
    rejected = f'A resposta completa é {resposta}.'
    rubrica = {'usou_contexto_cotidiano': contexto_referencia.lower() in chosen.lower(),
               'evitou_infantilizar': True, 'linguagem_simples': len(chosen.split()) <= 35,
               'nao_vazou_resposta': not any(c.isdigit() for c in chosen)}
    bruto = {'id': f"{exemplo['id']}:{tipo}", 'base_id': exemplo['base_id'], 'erro_tipo': tipo,
             'entrada': {'duvida': pergunta, 'contexto_cotidiano': contexto, 'historico_resumido': hist},
             'saida': {'dificuldade': dificuldade, 'verificacao_raciocinio': verificacao,
                       'acao': acao, 'resposta_esperada': resposta_esperada,
                       'justificativa_acao': justificativa},
             'rubrica': rubrica, 'dpo': {'chosen': chosen, 'rejected': rejected},
             'modo_geracao': exemplo['modo_geracao']}
    validado = Registro.model_validate(bruto)
    if not validado.rubrica.nao_vazou_resposta:
        raise ValueError(f"A resposta parece vazar em chosen: {validado.id}")
    return validado.model_dump()

if not ENTRADA.exists(): raise FileNotFoundError(f'Execute primeiro o notebook 03: {ENTRADA}')
exemplos = [json.loads(l) for l in ENTRADA.read_text(encoding='utf-8').splitlines() if l]
registros = [gerar_registro(x) for x in exemplos]
SAIDA.parent.mkdir(parents=True, exist_ok=True)
with SAIDA.open('w', encoding='utf-8') as arquivo:
    for registro in registros: arquivo.write(serializar_jsonl(registro) + '\n')
print(f'Registros validados: {len(registros)}; descartes: 0')

Registros validados: 18; descartes: 0


## Decisão pedagógica das ações

- `PERGUNTAR`: não há tentativa ou raciocínio suficiente; pergunta pelo primeiro passo.
- `DAR_DICA`: erro de operação, cálculo ou leitura; dá uma orientação específica sem resolver.
- `VALIDAR_PARCIALMENTE`: reconhece o que foi feito e convida a completar a etapa omitida.
- `PROPOR_EXERCICIO`: contas verificadas corretas; propõe prática nova, sem repetir a resposta.

O texto `rejected` entrega a resposta de referência deliberadamente para formar o par de preferência; `chosen` deve apoiar o próximo passo sem números de resposta.

In [7]:
salvos = [json.loads(l) for l in SAIDA.read_text(encoding='utf-8').splitlines() if l]
assert len(salvos) == len(registros)
assert all(set(r['rubrica'].values()) == {True} for r in salvos)
print('Total:', len(salvos), '| por erro:', {t: sum(r['erro_tipo'] == t for r in salvos) for t in sorted({r['erro_tipo'] for r in salvos})})
print('Amostras JSONL (uma por tipo disponível):')
vistos = set()
for r in salvos:
    if r['erro_tipo'] not in vistos:
        print(json.dumps(r, ensure_ascii=False))
        vistos.add(r['erro_tipo'])
    if len(vistos) == 3: break
print('Tipos mostrados:', sorted(vistos))

Total: 18 | por erro: {'ERRO_CALCULO': 4, 'ERRO_LEITURA': 4, 'ERRO_OPERACAO': 3, 'PASSO_OMITIDO': 3, 'SEM_ERRO': 4}
Amostras JSONL (uma por tipo disponível):
{"id": "train:0:ERRO_CALCULO", "base_id": "train:0", "erro_tipo": "ERRO_CALCULO", "entrada": {"duvida": "Problema: Natalia vendeu 48 grampos para amigas em abril. Em maio, ela vendeu a metade dessa quantidade. Quantos grampos ela vendeu no total em abril e maio?\nMinha tentativa foi 48/2 = 25. Estou no caminho certo?", "contexto_cotidiano": "Venda de acessórios de cabelo entre amigas.", "historico_resumido": "Exemplo sintético; não corresponde ao histórico de uma pessoa real."}, "saida": {"dificuldade": "Precisão do cálculo aritmético.", "verificacao_raciocinio": "A operação escolhida é adequada, mas o resultado informado não confere com a calculadora.", "acao": "DAR_DICA", "resposta_esperada": "Resposta de referência para uso interno do revisor: 72. Não mostrar ao estudante.", "justificativa_acao": "A tentativa contém uma operaçã